# 16 — Structured masking on the real GAVD training partitions

Scattered targets often retain nearby time points and landmarks. Does
removing those clues encourage a useful movement representation?
We extend Notebook 11 by inspecting connected regions, full landmark
trajectories and interior gaps on **every real outer-training clip**,
for five folds and five seeds. Each structure gets its own scattered
reference with exactly the same realized target count.

[VideoMAE, NeurIPS 2022, §3.3](https://arxiv.org/html/2203.12602) repeats
spatial masks across frames, motivating full landmark trajectories here.
[I-JEPA, CVPR 2023, §3](https://arxiv.org/html/2301.08243v3) predicts
image regions from surrounding context. [SLiM, §3.3](https://arxiv.org/html/2603.10648v3)
masks connected anatomical subsets over consecutive time spans. Our
fixed region size and duration isolate a simpler skeleton comparison;
these are adaptations, not reproductions of those complete recipes.

In [ ]:
from pathlib import Path
from dataclasses import asdict, replace
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from matplotlib_inline.backend_inline import set_matplotlib_formats

def locate_suite():
    for parent in (Path.cwd(), *Path.cwd().parents):
        for candidate in (parent, parent / "neurips-laterality"):
            if (candidate / "laterality_extensions/motion_structured_masks.py").is_file():
                return candidate.resolve()
    raise FileNotFoundError("Run from the research project directory.")

SUITE_ROOT = locate_suite()
if str(SUITE_ROOT) not in sys.path:
    sys.path.insert(0, str(SUITE_ROOT))
from laterality_extensions.motion_structured_masks import (
    StudyArm, mamp_logits, sample_study_mask, study_arms,
    paired_study_masks, context_cue_audit,
)
from laterality_extensions.comparative_masks import motion_scores
from notebook_progress import (
    NotebookTaskProgress, run_notebook_task, study_inputs_with_progress,
    audit_training_masks_with_progress, grid_status_with_progress,
    run_gavd_grid_with_progress, collect_gavd_grid_with_progress,
    evaluate_retained_motion_with_progress,
)
set_matplotlib_formats("svg", "png")
pd.set_option("display.precision", 3)

## 1. Load the real GAVD cohort and declare the full split/seed grid

Run cells in order in a Python kernel with the project's dependencies.
Long tasks use the shared `notebook_progress.py` wrapper: one updating
display shows the current stage, fold/seed, elapsed time and estimated
remaining time. Mask batches and optimizer updates appear within their
active stage. ETA adjusts as stages finish; their costs differ. Cached,
disabled, missing-input and failed tasks receive explicit status labels.
`DATA_MODE="gavd"` is the default. The helper below follows the same
preparation and source splitting functions as notebooks 01 and 02:

1. Verify an existing paper-profile cohort and split manifest by their
   content hashes. If absent, read the local GAVD pose archives and
   official annotations, apply the existing QC and target rules, and
   create those two artifacts. The first run takes longer.
2. The protocol fixes 642 pose archives and 666 annotations. If the local
   cache contains later additions, recover the original extraction
   generations only when their inventory **exactly** matches the locked
   count and SHA-256. Store verified copies under the paper artifact
   root. The original cache stays intact. A mismatch stops with an
   actionable error; it cannot silently switch to generated data.
3. The reference QC result is 625 clips from 93 source videos. Inputs have
   shape `[clips, 64, 33, 3]`; four prepared steps form each of 16 tokens
   per landmark. Naturally missing observations remain marked invalid.
   The target is a coordinate-derived bilateral movement contrast, not
   the dataset's condition annotation.
4. Reuse the five video-disjoint outer folds. Seeds 42--46 change
   initialization, source draws, augmentations and masks; they repeat
   the **same** train/test partitions. A video is held out exactly once
   per seed. There are 25 fold/seed combinations, not 25 independent
   datasets. Video separation does not establish subject separation.

The printed census makes train and test membership visible. Source IDs
and sequence IDs are retained in `inputs["memberships"]`. The reference
train/test clip counts are 436/189, 443/182, 553/72, 548/77 and 520/105.
Unequal clip counts are expected because entire videos stay together.

A small generated-data path remains available only through the explicit
`LATERALITY_MOTION_DATA_MODE=synthetic` software-check setting. It prints
its reduced scope and uses a separate artifact directory. It provides
no GAVD results. See [the run guide](docs/MOTION_GAVD_WORKFLOW.md) for
paths, environment settings, recovery and a notebook-by-notebook walkthrough.

In [ ]:
from laterality_extensions.motion_gavd import gavd_plan, readout_contrasts
DATA_MODE = os.getenv("LATERALITY_MOTION_DATA_MODE", "gavd")
FOLDS = (0, 1, 2, 3, 4)
SEEDS = (42, 43, 44, 45, 46)
EXPERIMENTS = tuple(os.getenv("LATERALITY_MOTION_EXPERIMENTS", "motion,regions").split(","))
CREATE_MISSING_INPUTS = True
DEVICE = os.getenv("LATERALITY_DEVICE", "auto")
# Same explicit training switch as Notebook 12; also accept the study-specific alias.
RUN_TRAINING = os.getenv("LATERALITY_MOTION_RUN_REAL",
                        os.getenv("LATERALITY_RESEARCH_RUN_REAL", "0")) == "1"
if DATA_MODE == "synthetic":
    FOLDS, SEEDS = (0,), (42,)
    print("EXPLICIT SYNTHETIC SOFTWARE CHECK: one fold/seed, one update, no GAVD evidence")
OUTPUT_ROOT = Path(os.getenv("LATERALITY_MOTION_OUTPUT_ROOT", str(SUITE_ROOT / "artifacts" /
    ("motion_structured" if DATA_MODE == "gavd" else "motion_structured_synthetic"))))
print(f"Mode={DATA_MODE}; folds={FOLDS}; seeds={SEEDS}; device={DEVICE}")
print(f"Training enabled={RUN_TRAINING}; outputs={OUTPUT_ROOT}")

In [ ]:
input_progress = NotebookTaskProgress("Dataset preparation and source splits", "stage")
inputs = study_inputs_with_progress(mode=DATA_MODE, folds=FOLDS, seeds=SEEDS,
    create_missing=CREATE_MISSING_INPUTS, progress=input_progress)
display(inputs["census"])
assert inputs["census"].source_overlap.eq(0).all()
display(inputs["memberships"].head(8))
if DATA_MODE == "gavd":
    cohort = inputs["cohort"]
    display(cohort.table.groupby("condition").agg(
        accepted_clips=("sequence_id", "size"), source_videos=("video_id", "nunique")))
    display(pd.Series({key: cohort.attrition[key] for key in
        ("input_sequences", "accepted_sequences", "accepted_sources", "excluded_sequences")}))
    print("Cohort:", cohort.cohort_digest)
    print("Split:", inputs["splits"]["split_digest"])
    print("Artifacts:", inputs["context"].artifact_root)

## 2. Define three separate prediction problems

| Experiment | Structured target on complete 64-step input | Nominal tokens | Visible context |
|---|---|---:|---|
| `regions` (primary) | Six connected landmarks for eight blocks | 48 | Other landmarks and times |
| `trajectories` (follow-up) | Three landmarks for all 16 blocks | 48 | Other landmark trajectories |
| `completion` (follow-up) | All 33 landmarks for four interior blocks | 132 | Earlier and later observations |

Region definitions use anatomical graph connections from Notebook 11.
Their union covers all 33 landmarks. Array-neighboring IDs do not define
a body region; head-to-body links are explicit masking connections.

Missing observations reduce realized counts. The structured mask keeps
its geometry, then its scattered reference hides exactly that many valid
tokens in the same clip. No missing coordinate becomes a target. We do
not trim full trajectories to fit a count. A structure with no feasible
target/context fails before optimization.

The two nominally 48-token structures need not realize equal counts with
missing data. Completion has a different budget. Compare each against
its own reference; a raw cross-experiment ranking confounds geometry
with the amount of hidden information.

In [ ]:
from laterality_extensions.comparative_masks import connected_region_bank
regions = connected_region_bank(tuple(range(33)), 6)
assert set().union(*map(set, regions)) == set(range(33))
print(f"{len(regions)} declared regions collectively cover all 33 landmarks.")
mask_progress = NotebookTaskProgress("Structured-mask audit", "fold/seed pass")
structure_audit = audit_training_masks_with_progress(inputs,
    experiments=("regions", "trajectories", "completion"), progress=mask_progress)
display(structure_audit["summary"])
assert structure_audit["per_clip"].groupby([
    "experiment", "fold", "seed", "sequence_id"]).hidden_tokens.nunique().eq(1).all()
print(f"Audited {len(structure_audit['per_clip']):,} training-clip mask draws.")

## 3. See what a real clip leaves available

Each row compares one structured family to its own uniform draw. Gray
marks natural missingness, blue visible context, and orange deliberate
targets. Panels identify the same first training clip selected before
outcome analysis. Geometry remains intact even when gray cells interrupt
its observed target support.

In [ ]:
from matplotlib.colors import ListedColormap
fig, axes = plt.subplots(3, 2, figsize=(11, 9), constrained_layout=True)
example_rows = []
for row, experiment in enumerate(("regions", "trajectories", "completion")):
    examples = [(name, value) for (e, name), value in structure_audit["examples"].items() if e == experiment]
    for ax, (name, example) in zip(axes[row], examples):
        state = np.where(example["valid"], 1, 0); state[example["mask"]] = 2
        ax.imshow(state.T, origin="lower", aspect="auto", vmin=0, vmax=2,
                  cmap=ListedColormap(["#d4d4d4", "#72a8cf", "#df9340"]))
        ax.set(title=f"{experiment}: {name}, K={example['mask'].sum()}",
               xlabel="Four-step block", ylabel="Landmark ID")
        example_rows.append({"experiment": experiment, "condition": name,
            **{k: example[k] for k in ("sequence_id", "source_id", "fold", "seed")},
            **context_cue_audit(example["mask"], example["valid"])})
display(fig); plt.close(fig)
display(pd.DataFrame(example_rows))

## 4. Quantify contextual cues across all folds and seeds

`temporal_bracket_fraction` counts targets with the same landmark visible
in **both immediately adjacent** time blocks. Full trajectories should
have zero such brackets. `visible_neighbor_fraction` counts targets
with at least one visible graph neighbor at the same time. A lower value
means fewer of these specific clues remain.

These are descriptive audits. Contextualized teacher features can
require more than coordinate interpolation; fewer clues can also make
asymmetric movement ambiguous. Neither fraction measures learned
semantics. Each fold/seed mean weights its training videos equally.
Averaging these dependent rows does not create independent observations.

In [ ]:
summary = structure_audit["summary"]
display(summary.groupby(["experiment", "condition"], sort=False).agg(
    smallest_realized_count=("hidden_min", "min"), largest_realized_count=("hidden_max", "max"),
    mean_hidden_fraction=("hidden_fraction", "mean"),
    temporal_brackets=("temporal_bracket_fraction", "mean"),
    visible_neighbors=("visible_neighbor_fraction", "mean")))
cue = summary.groupby(["experiment", "condition"], sort=False)[[
    "temporal_bracket_fraction", "visible_neighbor_fraction"]].mean()
ax = cue.plot.barh(figsize=(10, 4.5), xlim=(0, 1), title=f"{DATA_MODE.upper()}: available local cues")
ax.set_xlabel("Source-balanced fraction of hidden targets")
ax.figure.tight_layout(); display(ax.figure); plt.close(ax.figure)

## 5. Predeclare the next inference

Notebook 17's primary grid includes connected regions and the three
motion arms from Notebook 15. Trajectories and completion are declared
follow-ups; auditing them here does not claim they have been trained.
Add their experiment names to the configuration in both 17 and 18 to
conduct those comparisons.

If regions remove local cues yet fail to improve trained-over-initial
readout, cue removal alone is insufficient. If regions help against
their scattered reference and initial features, the trajectory follow-up
can distinguish spatial from temporal context effects. Avoid selecting
geometry by outer-test rank on this already inspected development cohort.

Interior gaps have context on both sides and test completion. Forecasting
requires the past-only inputs in Notebook 14. Removing prepared tokens
also differs from losing raw observations before interpolation and
normalization, addressed in Notebook 13. Continue with
[17](17_motion_and_structure_pretraining.ipynb) and
[18](18_motion_information_and_readout.ipynb).